In [1]:
import sqlite3
import pandas as pd
import numpy as np
import joblib
import copy
import time
from collections import deque
from itertools import combinations_with_replacement

conn = sqlite3.connect("../data/nfl.db")

xgb_model = joblib.load("../models/xgb_point_diff_v1.pkl")
feature_cols = joblib.load("../models/feature_cols_v1.pkl")

team_games = pd.read_sql("SELECT * FROM team_games", conn)
games = pd.read_sql("SELECT * FROM games", conn)

print("Model loaded. Expects", len(feature_cols), "features.")
print("team_games shape:", team_games.shape)
print("2026 games in schedule:", games[games["season"] == 2026].shape[0])

Model loaded. Expects 42 features.
team_games shape: (6600, 98)
2026 games in schedule: 272


In [2]:
stats_mask = team_games["passing_yards"].notna()

team_games["total_epa"] = np.nan
team_games["def_total_epa"] = np.nan

team_games.loc[stats_mask, "total_epa"] = (
    team_games.loc[stats_mask, "passing_epa"].fillna(0)
    + team_games.loc[stats_mask, "rushing_epa"].fillna(0)
    + team_games.loc[stats_mask, "receiving_epa"].fillna(0)
)

team_games.loc[stats_mask, "def_total_epa"] = (
    team_games.loc[stats_mask, "def_passing_epa"].fillna(0)
    + team_games.loc[stats_mask, "def_rushing_epa"].fillna(0)
    + team_games.loc[stats_mask, "def_receiving_epa"].fillna(0)
)

real_stats = team_games.dropna(subset=["total_epa"])
print("Most recent season with real EPA stats:", real_stats["season"].max())
print("Most recent week with real EPA stats:", real_stats[real_stats["season"] == real_stats["season"].max()]["week"].max())

Most recent season with real EPA stats: 2024
Most recent week with real EPA stats: 22


In [ ]:
team_games = team_games.sort_values(["team_id", "season", "week"]).reset_index(drop=True)

# points-based rolling: real data covers the full history 
points_stats = ["points_for", "points_against"]
grouped = team_games.groupby("team_id")

for col in points_stats:
    shifted = grouped[col].shift(1)
    team_games[f"{col}_roll3"] = shifted.groupby(team_games["team_id"]).transform(lambda x: x.rolling(3, min_periods=1).mean())
    team_games[f"{col}_roll8"] = shifted.groupby(team_games["team_id"]).transform(lambda x: x.rolling(8, min_periods=1).mean())
    team_games[f"{col}_season"] = shifted.groupby(team_games["season"].astype(str) + "_" + team_games["team_id"].astype(str)).transform(lambda x: x.expanding(min_periods=1).mean())

#  EPA/yardage rolling: compute on real data only then carry forward 
epa_stats = ["total_epa", "def_total_epa", "total_yards", "def_total_yards"]

for col in epa_stats:
    real_only = team_games[col].where(stats_mask)
    shifted = real_only.groupby(team_games["team_id"]).shift(1)

    team_games[f"{col}_roll3"] = shifted.groupby(team_games["team_id"]).transform(lambda x: x.rolling(3, min_periods=1).mean())
    team_games[f"{col}_roll8"] = shifted.groupby(team_games["team_id"]).transform(lambda x: x.rolling(8, min_periods=1).mean())
    team_games[f"{col}_season"] = shifted.groupby(team_games["season"].astype(str) + "_" + team_games["team_id"].astype(str)).transform(lambda x: x.expanding(min_periods=1).mean())

    # manually carry each teams last real rolling value forward onto future 
    for roll_col in [f"{col}_roll3", f"{col}_roll8", f"{col}_season"]:
        last_real_idx = team_games[stats_mask].groupby("team_id").tail(1).index
        last_real_values = team_games.loc[last_real_idx, ["team_id", roll_col]].set_index("team_id")[roll_col]
        future_mask = ~stats_mask
        team_games.loc[future_mask, roll_col] = team_games.loc[future_mask, "team_id"].map(last_real_values)

print("team_games shape after rolling features:", team_games.shape)

team_games shape after rolling features: (6600, 98)


In [ ]:
teams = pd.read_sql("SELECT team_abbr, team_id FROM teams", conn)
abbr_to_id = dict(zip(teams["team_abbr"], teams["team_id"]))

teams_full = pd.read_sql("SELECT * FROM teams", conn)
team_conf_div = teams_full[["team_id", "team_conf", "team_division"]].drop_duplicates(subset="team_id")

# current-abbreviation lookup, based on each team's most recent season on record
recent_abbr = team_games.sort_values("season").groupby("team_id")["team_abbr_current"].last()
team_abbr_lookup = recent_abbr.to_dict()

# manual overrides for relocated franchises the automated lookup can get wrong
abbr_overrides = {
    2510: "LAR",             # Rams: St. Louis -> Los Angeles
    abbr_to_id["LV"]: "LV",  # Raiders: Oakland -> Las Vegas
}
team_abbr_lookup.update(abbr_overrides)

print("Rams:", team_abbr_lookup.get(2510))
print("Raiders:", team_abbr_lookup.get(abbr_to_id["LV"]))

Rams: LAR
Raiders: LV
KC: KC


In [ ]:
POWER_RANKINGS = {
    "LA": 1, "DEN": 2, "SEA": 3, "PHI": 4, "BUF": 5, "BAL": 6, "JAX": 7, "CIN": 8,
    "HOU": 9, "CHI": 10, "DAL": 11, "NE": 12, "GB": 13, "DET": 14, "KC": 15, "SF": 16,
    "LAC": 17, "MIN": 18, "IND": 19, "PIT": 20, "NO": 21, "TB": 22, "CAR": 23, "TEN": 24,
    "ATL": 25, "ARI": 26, "NYG": 27, "CLE": 28, "WAS": 29, "NYJ": 30, "LV": 31, "MIA": 32,
}

# strength: +1.0 (rank 1, best) to -1.0 (rank 32, worst), 0 at the midpoint (16.5)
# This is in orde r to make power rankings not insanely overpowered in sim aka balancing
team_strength = {}
for abbr, rank in POWER_RANKINGS.items():
    tid = abbr_to_id[abbr]
    team_strength[tid] = (16.5 - rank) / 15.5

# quick sanity check
print("Rams (rank 1) strength:", round(team_strength[abbr_to_id["LA"]], 3))
print("Dolphins (rank 32) strength:", round(team_strength[abbr_to_id["MIA"]], 3))
print("Chiefs (rank 15) strength:", round(team_strength[abbr_to_id["KC"]], 3))

Rams (rank 1) strength: 1.0
Dolphins (rank 32) strength: -1.0
Chiefs (rank 15) strength: 0.097


In [ ]:
MAX_WEEK1_NUDGE = 3.0      # points at full strength gap (rank 1 vs rank 32), a team's game 1
MAX_PERMANENT_PULL = 1.0   # points at full strength gap, every game, all season -- never fully disappears

def get_power_nudge(team_id, opp_id, team_states):
    """Point-differential nudge for team_id in a matchup vs opp_id.
    Combines a decaying Week-1-era nudge with a small permanent pull.
    Both are added to the model's predicted differential BEFORE random noise,
    so they shift the center of the outcome distribution without touching its
    spread -- a #1 team can still lose, a #32 team can still surprise.
    """
    strength_gap = team_strength.get(team_id, 0) - team_strength.get(opp_id, 0)  # -2 to +2

    games_played = len(team_states[team_id]["points_for"]["season_vals"])
    decay = max(0, 1 - games_played / 9)  # fully faded out by a team's 9th game

    week1_component = MAX_WEEK1_NUDGE * (strength_gap / 2) * decay
    permanent_component = MAX_PERMANENT_PULL * (strength_gap / 2)

    return week1_component + permanent_component

In [7]:
stat_names = ["points_for", "points_against", "total_epa", "def_total_epa", "total_yards", "def_total_yards"]

def init_team_states(team_games, teams_list):
    states = {}
    for tid in teams_list:
        team_hist = team_games[team_games["team_id"] == tid].sort_values(["season", "week"])
        states[tid] = {}
        for stat in stat_names:
            real_vals = team_hist[team_hist[stat].notna()][stat].tolist()
            last3 = real_vals[-3:] if len(real_vals) >= 1 else [0]
            last8 = real_vals[-8:] if len(real_vals) >= 1 else [0]
            states[tid][stat] = {
                "roll3_deque": deque(last3, maxlen=3),
                "roll8_deque": deque(last8, maxlen=8),
                "season_vals": []  # resets empty -- 2026 season-to-date starts fresh
            }
    return states

all_team_ids = team_games["team_id"].unique()
team_states = init_team_states(team_games, all_team_ids)

kc_id = abbr_to_id["KC"]
print("KC points_for roll3 seed:", list(team_states[kc_id]["points_for"]["roll3_deque"]))
print("KC points_for roll8 seed:", list(team_states[kc_id]["points_for"]["roll8_deque"]))

KC points_for roll3 seed: [9.0, 13.0, 12.0]
KC points_for roll8 seed: [19.0, 23.0, 28.0, 10.0, 13.0, 9.0, 13.0, 12.0]


In [ ]:
MODEL_MAE = 10.3  # real games deviate from the models point-diff prediction by roughly this much on average

def get_rolling_avg(team_states, tid, stat, window):
    if window == "roll3":
        vals = list(team_states[tid][stat]["roll3_deque"])
    elif window == "roll8":
        vals = list(team_states[tid][stat]["roll8_deque"])
    elif window == "season":
        vals = team_states[tid][stat]["season_vals"]
        if len(vals) == 0:
            vals = list(team_states[tid][stat]["roll8_deque"])  # season hasn't started yet -- use roll8 as proxy
    return np.mean(vals) if len(vals) > 0 else 0

def build_features(team_states, team_id, opp_id, is_home, div_game, rest_days, opp_rest_days,
                    temp, wind, is_indoor, qb_out, rb_out, wr_out):
    row = {}
    row["div_game"] = div_game
    row["temp"] = temp
    row["wind"] = wind
    row["rest_days"] = rest_days
    row["opp_rest_days"] = opp_rest_days
    row["is_home"] = is_home
    row["rest_advantage"] = rest_days - opp_rest_days
    row["short_week"] = int(rest_days < 6)
    row["is_indoor"] = is_indoor
    row["qb_out"] = qb_out
    row["rb_starter_out"] = rb_out
    row["wr_starter_out"] = wr_out

    for stat in ["points_for", "points_against", "total_epa", "def_total_epa", "total_yards", "def_total_yards"]:
        row[f"{stat}_roll3"] = get_rolling_avg(team_states, team_id, stat, "roll3")
        row[f"{stat}_roll8"] = get_rolling_avg(team_states, team_id, stat, "roll8")
        row[f"{stat}_season"] = get_rolling_avg(team_states, team_id, stat, "season")

    row["opp_def_epa_roll3"] = get_rolling_avg(team_states, opp_id, "def_total_epa", "roll3")
    row["opp_def_epa_roll8"] = get_rolling_avg(team_states, opp_id, "def_total_epa", "roll8")
    row["opp_def_epa_season"] = get_rolling_avg(team_states, opp_id, "def_total_epa", "season")

    row["opp_off_epa_roll3"] = get_rolling_avg(team_states, opp_id, "total_epa", "roll3")
    row["opp_off_epa_roll8"] = get_rolling_avg(team_states, opp_id, "total_epa", "roll8")
    row["opp_off_epa_season"] = get_rolling_avg(team_states, opp_id, "total_epa", "season")

    row["adj_epa_roll3"] = row["total_epa_roll3"] - row["opp_def_epa_roll3"]
    row["adj_epa_roll8"] = row["total_epa_roll8"] - row["opp_def_epa_roll8"]
    row["adj_epa_season"] = row["total_epa_season"] - row["opp_def_epa_season"]

    row["adj_def_epa_roll3"] = row["opp_off_epa_roll3"] - row["def_total_epa_roll3"]
    row["adj_def_epa_roll8"] = row["opp_off_epa_roll8"] - row["def_total_epa_roll8"]
    row["adj_def_epa_season"] = row["opp_off_epa_season"] - row["def_total_epa_season"]

    return row

In [ ]:
def predict_game(model, feature_cols, team_states, team_id, opp_id, is_home, div_game,
                  rest_days, opp_rest_days, temp, wind, is_indoor, qb_out, rb_out, wr_out,
                  mae=MODEL_MAE):
    row = build_features(team_states, team_id, opp_id, is_home, div_game, rest_days, opp_rest_days,
                          temp, wind, is_indoor, qb_out, rb_out, wr_out)

    X = pd.DataFrame([row])[feature_cols]  # enforce exact column order the model expects
    predicted_diff = model.predict(X)[0]

    # power rankings nudge: small decaying and permanent shift toward the higher-ranked team
    predicted_diff += get_power_nudge(team_id, opp_id, team_states)

    # real games deviate from the models prediction by roughly MODEL_MAE on average
    noise = np.random.normal(loc=0, scale=mae)
    simulated_diff = predicted_diff + noise

    return simulated_diff, predicted_diff

# quick test
sim_diff, pred_diff = predict_game(
    xgb_model, feature_cols, team_states, team_id=kc_id, opp_id=abbr_to_id["BAL"],
    is_home=1, div_game=0, rest_days=7, opp_rest_days=7,
    temp=70, wind=0, is_indoor=0, qb_out=0, rb_out=0, wr_out=0
)
print("Model's raw + power-nudged predicted differential:", round(pred_diff, 2))
print("Simulated differential (with randomness):", round(sim_diff, 2))

Model's raw + power-nudged predicted differential: -5.51
Simulated differential (with randomness): 1.89


In [ ]:
real_games = team_games[team_games["points_for"].notna()]
team_scoring_std = real_games.groupby("team_id")["points_for"].std().to_dict()
team_scoring_std_default = real_games["points_for"].std()

def get_team_scoring_std(tid):
    val = team_scoring_std.get(tid, np.nan)
    return val if not np.isnan(val) else team_scoring_std_default

LEAGUE_AVG_TOTAL = 45.67
LEAGUE_STD_TOTAL = 13.88
TD_PROBABILITY = 0.58  # probability a scoring drive ends in a TD (7) rather than FG (3)

def simulate_score_from_avg(avg_points):
    points_per_drive = TD_PROBABILITY * 7 + (1 - TD_PROBABILITY) * 3
    est_drives = max(round(avg_points / points_per_drive), 0)
    est_drives = max(round(np.random.normal(est_drives, 1.2)), 0)  # round(), NOT int() -- int() truncates and biases low

    total = 0
    for _ in range(est_drives):
        total += 7 if np.random.random() < TD_PROBABILITY else 3
    return total

# verify by feeding the real league average in should return roughly similar average
test_outputs = [simulate_score_from_avg(22.9) for _ in range(500)]
print("Input average: 22.9  ->  Output average:", round(np.mean(test_outputs), 2))

Input average: 22.9  ->  Output average: 20.99


In [ ]:
def simulate_game_result(team_states, team_id, opp_id, simulated_diff):
    team_off_avg = get_rolling_avg(team_states, team_id, "points_for", "roll8")
    team_def_avg = get_rolling_avg(team_states, team_id, "points_against", "roll8")
    opp_off_avg = get_rolling_avg(team_states, opp_id, "points_for", "roll8")
    opp_def_avg = get_rolling_avg(team_states, opp_id, "points_against", "roll8")

    # each teams expected total is the SUM of both teams expected points (not an average of 4 numbers)
    home_expected = (team_off_avg + opp_def_avg) / 2
    away_expected = (opp_off_avg + team_def_avg) / 2
    expected_total = home_expected + away_expected
    blended_total = (expected_total + LEAGUE_AVG_TOTAL) / 2

    team_std = get_team_scoring_std(team_id)
    opp_std = get_team_scoring_std(opp_id)
    combined_std = np.sqrt(team_std**2 + opp_std**2)
    total_points = max(blended_total + np.random.normal(0, combined_std), 6)

    # simulate each teams score independently using realistic TD/FG-shaped scoring
    team_points = simulate_score_from_avg(home_expected)
    opp_points = simulate_score_from_avg(away_expected)

    # nudge toward the models predicted differential since that remains our best signal of who wins and by how much
    actual_diff = team_points - opp_points
    diff_error = simulated_diff - actual_diff
    adjustment = round(diff_error / 2)
    team_points = max(team_points + adjustment, 0)
    opp_points = max(opp_points - adjustment, 0)

    team_epa = simulated_diff * 0.5 + np.random.normal(0, 8)
    opp_epa = -simulated_diff * 0.5 + np.random.normal(0, 8)
    team_yards = 350 + simulated_diff * 3 + np.random.normal(0, 30)
    opp_yards = 350 - simulated_diff * 3 + np.random.normal(0, 30)

    return {
        "points_for": team_points, "points_against": opp_points,
        "total_epa": team_epa, "def_total_epa": opp_epa,
        "total_yards": max(team_yards, 0), "def_total_yards": max(opp_yards, 0)
    }

def update_team_state(team_states, tid, result):
    for stat in stat_names:
        val = result[stat]
        team_states[tid][stat]["roll3_deque"].append(val)
        team_states[tid][stat]["roll8_deque"].append(val)
        team_states[tid][stat]["season_vals"].append(val)

In [12]:
schedule_2026 = games[games["season"] == 2026].copy()
schedule_2026["home_id"] = schedule_2026["home_team"].map(abbr_to_id)
schedule_2026["away_id"] = schedule_2026["away_team"].map(abbr_to_id)
schedule_2026 = schedule_2026.sort_values(["week", "game_id"]).reset_index(drop=True)

stadium_roof = team_games.dropna(subset=["roof"]).groupby("stadium")["roof"].agg(lambda x: x.mode()[0])

def get_roof(row):
    if pd.notna(row["roof"]):
        return row["roof"]
    return stadium_roof.get(row["stadium"], "outdoors")

schedule_2026["roof"] = schedule_2026.apply(get_roof, axis=1)

print(schedule_2026.shape)
print(schedule_2026["roof"].value_counts(dropna=False))

(272, 27)
outdoors    187
dome         52
closed       33
Name: roof, dtype: int64


In [ ]:
pgs = pd.read_sql("SELECT * FROM player_game_stats", conn)
pgs_all = pgs.copy()
pgs_all["team_id"] = pgs_all["recent_team"].map(abbr_to_id)

def compute_rank_shares(pgs_df, stat_col, position_filter=None):
    df = pgs_df.copy()
    if position_filter:
        df = df[df["position"].isin(position_filter)]
    season_totals = df.groupby(["team_id", "season", "player_id"])[stat_col].sum().reset_index()
    season_totals["rank"] = season_totals.groupby(["team_id", "season"])[stat_col].rank(method="first", ascending=False)
    team_season_totals = season_totals.groupby(["team_id", "season"])[stat_col].transform("sum")
    season_totals["share"] = season_totals[stat_col] / team_season_totals
    return season_totals.groupby("rank")["share"].mean()

rush_shares = compute_rank_shares(pgs_all, "rushing_yards", position_filter=["RB"])
wr_shares = compute_rank_shares(pgs_all, "receiving_yards", position_filter=["WR"])
te_shares = compute_rank_shares(pgs_all, "receiving_yards", position_filter=["TE"])
pass_shares = compute_rank_shares(pgs_all, "passing_yards", position_filter=["QB"])

wr_te_split = pgs_all[pgs_all["position"].isin(["WR", "TE"])].groupby(["team_id", "season", "position"])["receiving_yards"].sum().reset_index()
position_totals = wr_te_split.groupby("position")["receiving_yards"].sum()
position_share = position_totals / position_totals.sum()

# current depth charts split by skill position
depth_current = pd.read_sql("SELECT * FROM depth_charts_current", conn)
depth_current["team_id"] = depth_current["team"].map(abbr_to_id)
skill_positions = ["QB", "RB", "WR", "TE"]
depth_current_skill = depth_current[depth_current["pos_abb"].isin(skill_positions)].copy()

current_qbs = depth_current_skill[depth_current_skill["pos_abb"] == "QB"]
current_rbs = depth_current_skill[depth_current_skill["pos_abb"] == "RB"]
current_wrs = depth_current_skill[depth_current_skill["pos_abb"] == "WR"]
current_tes = depth_current_skill[depth_current_skill["pos_abb"] == "TE"]

# real team-level pass/rush split (2020-2024) used to convert team total yards into pass/rush pools
real_2020_2024 = team_games[(team_games["season"].between(2020, 2024)) & (team_games["passing_yards"].notna())]
team_split = real_2020_2024.groupby("team_id").agg(
    total_pass_yards=("passing_yards", "sum"),
    total_rush_yards=("rushing_yards", "sum")
)
team_split["pass_share"] = team_split["total_pass_yards"] / (team_split["total_pass_yards"] + team_split["total_rush_yards"])

# real team-level rush/pass TD split (2020-2024) -- based on real TD COUNTS, not TDs-per-yard
# (TDs-per-yard incorrectly implies a 50/50 split; real counts show the true 30/70 rush/pass split)
real_recent_full = pgs[pgs["season"].between(2020, 2024)].copy()
real_recent_full["team_id"] = real_recent_full["recent_team"].map(abbr_to_id)
team_td_counts = real_recent_full.groupby("team_id").agg(
    total_pass_tds=("passing_tds", "sum"),
    total_rush_tds=("rushing_tds", "sum")
)
team_td_counts["rush_td_share_correct"] = team_td_counts["total_rush_tds"] / (team_td_counts["total_rush_tds"] + team_td_counts["total_pass_tds"])

# real per-team rates: yards per carry/target/attempt, completion %, TD rates -- used to derive carries/receptions/attempts
real_recent = pgs[pgs["season"].between(2020, 2024)].copy()
real_recent["team_id"] = real_recent["recent_team"].map(abbr_to_id)

rush_rates = real_recent.groupby("team_id").apply(
    lambda x: pd.Series({
        "yards_per_carry": x["rushing_yards"].sum() / x["carries"].sum() if x["carries"].sum() > 0 else 4.2,
        "rush_td_rate": x["rushing_tds"].sum() / x["rushing_yards"].sum() if x["rushing_yards"].sum() > 0 else 0.01
    })
)
pass_rates = real_recent.groupby("team_id").apply(
    lambda x: pd.Series({
        "comp_pct": x["completions"].sum() / x["attempts"].sum() if x["attempts"].sum() > 0 else 0.63,
        "pass_td_rate": x["passing_tds"].sum() / x["passing_yards"].sum() if x["passing_yards"].sum() > 0 else 0.02
    })
)
rec_rates = real_recent.groupby("team_id").apply(
    lambda x: pd.Series({
        "yards_per_target": x["receiving_yards"].sum() / x["targets"].sum() if x["targets"].sum() > 0 else 7.5,
        "catch_rate": x["receptions"].sum() / x["targets"].sum() if x["targets"].sum() > 0 else 0.63,
    })
)

# real team-level avg offensive TDs/game (2020-2024) -- used by estimate_tds_from_score_v5
team_td_rates = team_games[
    (team_games["season"].between(2020, 2024)) & (team_games["passing_tds"].notna())
].groupby("team_id").apply(
    lambda x: (x["passing_tds"] + x["rushing_tds"]).mean()
).rename("avg_tds_per_game")

print("Player distribution reference tables built.")

Player distribution reference tables built.


In [14]:
def sample_player_shares_v2(rank_shares_table, num_players, noise_std=0.35, bench_zero_prob=0.35, zero_eligible_ranks=None):
    base_shares = np.array([rank_shares_table.get(float(r), 0) for r in range(1, num_players + 1)])
    noisy_shares = base_shares * np.exp(np.random.normal(0, noise_std, size=num_players))

    zero_eligible_ranks = zero_eligible_ranks or []
    for i in range(num_players):
        rank = i + 1
        if rank in zero_eligible_ranks and np.random.random() < bench_zero_prob:
            noisy_shares[i] = 0.001

    return noisy_shares / noisy_shares.sum()

def distribute_stats_capped_noisy_v2(team_total, current_depth_team, rank_shares_table, max_players,
                                      noise_std=0.35, zero_eligible_ranks=None, bench_zero_prob=0.35):
    eligible = current_depth_team[current_depth_team["pos_rank"] <= max_players].copy()
    n = len(eligible)
    if n == 0:
        return pd.DataFrame(columns=["player_name", "gsis_id", "rank", "share", "simulated_stat"])

    noisy_shares = sample_player_shares_v2(rank_shares_table, n, noise_std, bench_zero_prob, zero_eligible_ranks)

    result = []
    for i, (_, player) in enumerate(eligible.iterrows()):
        player_stat = team_total * noisy_shares[i]
        result.append({"player_name": player["player_name"], "gsis_id": player["gsis_id"],
                        "rank": player["pos_rank"], "share": noisy_shares[i], "simulated_stat": round(player_stat, 1)})
    return pd.DataFrame(result)

def distribute_receiving_capped_noisy_v2(team_total_rec_yards, current_wrs, current_tes, wr_rank_shares, te_rank_shares,
                                          position_split, max_wr=4, max_te=2, noise_std=0.35):
    wr_pool = team_total_rec_yards * position_split["WR"]
    te_pool = team_total_rec_yards * position_split["TE"]
    # WR4 (rank 4) sometimes zeroed out; TE2 (rank 2) sometimes zeroed out -- matches real bench-tier usage patterns
    wr_results = distribute_stats_capped_noisy_v2(wr_pool, current_wrs, wr_rank_shares, max_wr, noise_std, zero_eligible_ranks=[4], bench_zero_prob=0.35)
    te_results = distribute_stats_capped_noisy_v2(te_pool, current_tes, te_rank_shares, max_te, noise_std, zero_eligible_ranks=[2], bench_zero_prob=0.35)
    return pd.concat([wr_results, te_results], ignore_index=True)

def estimate_tds_from_score_v5(team_score, tid):
    """Reverse-engineers a plausible TD/FG split from a final score, sampling
    the TD count from this team's real historical TD rate rather than just
    greedily maximizing TDs."""
    if team_score == 0:
        return 0, 0
    avg_tds = team_td_rates.get(tid, team_td_rates.mean())
    tds = np.random.poisson(avg_tds)
    max_tds_possible = team_score // 6
    tds = min(tds, max_tds_possible)
    remaining = team_score - (tds * 7)
    while remaining < 0 and tds > 0:
        tds -= 1
        remaining = team_score - (tds * 7)
    fgs = max(remaining // 3, 0)
    return int(tds), int(fgs)

In [ ]:
def simulate_one_season_full(team_states_init, schedule_2026):
    team_states = copy.deepcopy(team_states_init)  # never mutate the team_states
    game_results = []
    player_stat_rows = []

    for _, game in schedule_2026.iterrows():
        home_id = game["home_id"]
        away_id = game["away_id"]

        is_indoor = 1 if game["roof"] in ["dome", "closed"] else 0
        temp = 70 if is_indoor else 60
        wind = 0 if is_indoor else 7
        div_game = game["div_game"]
        rest_days = 7
        opp_rest_days = 7
        qb_out, rb_out, wr_out = 0, 0, 0  # future injuries unknowable -- v1 simplification

        sim_diff, pred_diff = predict_game(
            xgb_model, feature_cols, team_states, team_id=home_id, opp_id=away_id,
            is_home=1, div_game=div_game, rest_days=rest_days, opp_rest_days=opp_rest_days,
            temp=temp, wind=wind, is_indoor=is_indoor, qb_out=qb_out, rb_out=rb_out, wr_out=wr_out
        )
        result = simulate_game_result(team_states, home_id, away_id, sim_diff)

        game_results.append({
            "game_id": game["game_id"], "week": game["week"],
            "home_id": home_id, "away_id": away_id,
            "home_score": result["points_for"], "away_score": result["points_against"],
            "home_epa": result["total_epa"], "away_epa": result["def_total_epa"],
            "home_yards": result["total_yards"], "away_yards": result["def_total_yards"],
        })

        for tid, team_yards, team_score in [(home_id, result["total_yards"], result["points_for"]),
                                              (away_id, result["def_total_yards"], result["points_against"])]:
            pass_share = team_split["pass_share"].get(tid, team_split["pass_share"].mean())
            rush_share = 1 - pass_share
            team_pass_yards = team_yards * pass_share
            team_rush_yards = team_yards * rush_share
            team_rec_yards = team_pass_yards

            qb_rows = distribute_stats_capped_noisy_v2(team_pass_yards, current_qbs[current_qbs["team_id"] == tid], pass_shares, max_players=1, noise_std=0.2)
            rb_rows = distribute_stats_capped_noisy_v2(team_rush_yards, current_rbs[current_rbs["team_id"] == tid], rush_shares, max_players=2, noise_std=0.35)
            rec_rows = distribute_receiving_capped_noisy_v2(team_rec_yards,
                                                             current_wrs[current_wrs["team_id"] == tid],
                                                             current_tes[current_tes["team_id"] == tid],
                                                             wr_shares, te_shares, position_share, max_wr=4, max_te=2, noise_std=0.35)

            total_tds, total_fgs = estimate_tds_from_score_v5(team_score, tid)
            rush_td_share = team_td_counts["rush_td_share_correct"].get(tid, team_td_counts["rush_td_share_correct"].mean())
            rush_tds_target = round(total_tds * rush_td_share)
            pass_tds_target = total_tds - rush_tds_target

            ypc = rush_rates.loc[tid, "yards_per_carry"]
            ypt_catch = rec_rates.loc[tid, "yards_per_target"] / rec_rates.loc[tid, "catch_rate"]
            comp_pct = pass_rates.loc[tid, "comp_pct"]

            rb_rows = rb_rows.copy()
            if len(rb_rows) > 0:
                rb_rows["carries"] = rb_rows["simulated_stat"].apply(lambda y: max(round(y / ypc), 1) if ypc > 0 else 1)
                rb_weights = (rb_rows["simulated_stat"] / rb_rows["simulated_stat"].sum()).values
                rb_rows["tds"] = np.random.multinomial(rush_tds_target, rb_weights) if rush_tds_target > 0 else [0] * len(rb_rows)
            else:
                rb_rows["carries"], rb_rows["tds"] = [], []

            rec_rows = rec_rows.copy()
            if len(rec_rows) > 0:
                rec_rows["receptions"] = rec_rows["simulated_stat"].apply(lambda y: max(round(y / ypt_catch), 1) if ypt_catch > 0 else 1)
                rec_weights = (rec_rows["simulated_stat"] / rec_rows["simulated_stat"].sum()).values
                rec_rows["tds"] = np.random.multinomial(pass_tds_target, rec_weights) if pass_tds_target > 0 else [0] * len(rec_rows)
                total_receptions_this_team = rec_rows["receptions"].sum()
            else:
                rec_rows["receptions"], rec_rows["tds"] = [], []
                total_receptions_this_team = 0

            qb_rows = qb_rows.copy()
            if len(qb_rows) > 0:
                qb_rows["completions"] = total_receptions_this_team
                qb_rows["attempts"] = max(round(total_receptions_this_team / comp_pct), total_receptions_this_team) if comp_pct > 0 else total_receptions_this_team
                qb_rows["tds"] = pass_tds_target

            for df, stat_type in [(qb_rows, "passing"), (rb_rows, "rushing"), (rec_rows, "receiving")]:
                df = df.copy()
                df["stat_type"] = stat_type
                df["team_id"] = tid
                df["game_id"] = game["game_id"]
                df["week"] = game["week"]
                player_stat_rows.append(df)

        update_team_state(team_states, home_id, result)
        update_team_state(team_states, away_id, {
            "points_for": result["points_against"], "points_against": result["points_for"],
            "total_epa": result["def_total_epa"], "def_total_epa": result["total_epa"],
            "total_yards": result["def_total_yards"], "def_total_yards": result["total_yards"]
        })

    games_df = pd.DataFrame(game_results)
    players_df = pd.concat(player_stat_rows, ignore_index=True)
    return games_df, players_df

In [16]:
def simulate_one_season_lite(team_states_init, schedule_2026):
    team_states = copy.deepcopy(team_states_init)
    game_results = []

    for _, game in schedule_2026.iterrows():
        home_id = game["home_id"]
        away_id = game["away_id"]

        is_indoor = 1 if game["roof"] in ["dome", "closed"] else 0
        temp = 70 if is_indoor else 60
        wind = 0 if is_indoor else 7
        div_game = game["div_game"]
        rest_days = 7
        opp_rest_days = 7
        qb_out, rb_out, wr_out = 0, 0, 0

        sim_diff, pred_diff = predict_game(
            xgb_model, feature_cols, team_states, team_id=home_id, opp_id=away_id,
            is_home=1, div_game=div_game, rest_days=rest_days, opp_rest_days=opp_rest_days,
            temp=temp, wind=wind, is_indoor=is_indoor, qb_out=qb_out, rb_out=rb_out, wr_out=wr_out
        )
        result = simulate_game_result(team_states, home_id, away_id, sim_diff)

        game_results.append({
            "game_id": game["game_id"], "week": game["week"],
            "home_id": home_id, "away_id": away_id,
            "home_score": result["points_for"], "away_score": result["points_against"],
        })

        update_team_state(team_states, home_id, result)
        update_team_state(team_states, away_id, {
            "points_for": result["points_against"], "points_against": result["points_for"],
            "total_epa": result["def_total_epa"], "def_total_epa": result["total_epa"],
            "total_yards": result["def_total_yards"], "def_total_yards": result["total_yards"]
        })

    return pd.DataFrame(game_results)

start_time = time.time()
test_lite = simulate_one_season_lite(team_states, schedule_2026)
print(f"Lite version: {time.time() - start_time:.2f} sec/season")

Lite version: 1.77 sec/season


In [17]:
def get_playoff_seeds(season_group):
    results = []
    for conf in ["AFC", "NFC"]:
        conf_teams = season_group[season_group["team_conf"] == conf].copy()
        conf_teams = conf_teams.sort_values(["wins", "avg_points"], ascending=[False, False])

        division_winners = conf_teams.groupby("team_division").head(1).copy()
        division_winners = division_winners.sort_values(["wins", "avg_points"], ascending=[False, False])
        division_winners["seed"] = range(1, len(division_winners) + 1)

        remaining = conf_teams[~conf_teams["team_id"].isin(division_winners["team_id"])]
        wild_cards = remaining.sort_values(["wins", "avg_points"], ascending=[False, False]).head(3).copy()
        wild_cards["seed"] = range(len(division_winners) + 1, len(division_winners) + 1 + len(wild_cards))

        conf_playoff_teams = pd.concat([division_winners, wild_cards])
        conf_playoff_teams["conf"] = conf
        results.append(conf_playoff_teams)

    return pd.concat(results)

def break_playoff_tie(result):
    """NFL playoff games can't end in a tie -- simulate a simple OT score if tied."""
    if result["points_for"] == result["points_against"]:
        ot_score = np.random.choice([3, 6, 7, 8], p=[0.55, 0.05, 0.30, 0.10])
        if np.random.random() < 0.5:
            result["points_for"] += ot_score
        else:
            result["points_against"] += ot_score
    return result

In [18]:
def simulate_playoff_game(team_states, team_a_id, team_b_id, neutral_site=False):
    is_home_a = 0 if neutral_site else 1
    sim_diff, _ = predict_game(
        xgb_model, feature_cols, team_states, team_id=team_a_id, opp_id=team_b_id,
        is_home=is_home_a, div_game=0, rest_days=7, opp_rest_days=7,
        temp=70, wind=0, is_indoor=1, qb_out=0, rb_out=0, wr_out=0
    )
    result = simulate_game_result(team_states, team_a_id, team_b_id, sim_diff)
    result = break_playoff_tie(result)
    winner = team_a_id if result["points_for"] > result["points_against"] else team_b_id

    update_team_state(team_states, team_a_id, result)
    update_team_state(team_states, team_b_id, {
        "points_for": result["points_against"], "points_against": result["points_for"],
        "total_epa": result["def_total_epa"], "def_total_epa": result["total_epa"],
        "total_yards": result["def_total_yards"], "def_total_yards": result["total_yards"]
    })
    return winner

def simulate_conference_bracket(team_states, seeds_dict):
    wc_winner_2_7 = simulate_playoff_game(team_states, seeds_dict[2], seeds_dict[7])
    wc_winner_3_6 = simulate_playoff_game(team_states, seeds_dict[3], seeds_dict[6])
    wc_winner_4_5 = simulate_playoff_game(team_states, seeds_dict[4], seeds_dict[5])

    wc_winners = [wc_winner_2_7, wc_winner_3_6, wc_winner_4_5]
    wc_seeds = {seeds_dict[2]: 2, seeds_dict[3]: 3, seeds_dict[4]: 4,
                seeds_dict[5]: 5, seeds_dict[6]: 6, seeds_dict[7]: 7}

    wc_winners_sorted = sorted(wc_winners, key=lambda t: wc_seeds[t])
    opponent_for_1 = wc_winners_sorted[-1]
    other_two = [t for t in wc_winners if t != opponent_for_1]

    div_winner_1 = simulate_playoff_game(team_states, seeds_dict[1], opponent_for_1)
    if wc_seeds[other_two[0]] < wc_seeds[other_two[1]]:
        div_winner_2 = simulate_playoff_game(team_states, other_two[0], other_two[1])
    else:
        div_winner_2 = simulate_playoff_game(team_states, other_two[1], other_two[0])

    return simulate_playoff_game(team_states, div_winner_1, div_winner_2)

def simulate_full_playoffs(team_states, playoff_seeds_this_season):
    afc_seeds = playoff_seeds_this_season[playoff_seeds_this_season["conf"] == "AFC"].set_index("seed")["team_id"].to_dict()
    nfc_seeds = playoff_seeds_this_season[playoff_seeds_this_season["conf"] == "NFC"].set_index("seed")["team_id"].to_dict()

    afc_champion = simulate_conference_bracket(team_states, afc_seeds)
    nfc_champion = simulate_conference_bracket(team_states, nfc_seeds)
    super_bowl_winner = simulate_playoff_game(team_states, afc_champion, nfc_champion, neutral_site=True)

    return super_bowl_winner, afc_champion, nfc_champion

In [19]:
def simulate_playoff_game_full(team_states, team_a_id, team_b_id, round_name, neutral_site=False):
    is_home_a = 0 if neutral_site else 1
    sim_diff, _ = predict_game(
        xgb_model, feature_cols, team_states, team_id=team_a_id, opp_id=team_b_id,
        is_home=is_home_a, div_game=0, rest_days=7, opp_rest_days=7,
        temp=70, wind=0, is_indoor=1, qb_out=0, rb_out=0, wr_out=0
    )
    result = simulate_game_result(team_states, team_a_id, team_b_id, sim_diff)
    result = break_playoff_tie(result)
    winner = team_a_id if result["points_for"] > result["points_against"] else team_b_id

    game_record = {
        "round": round_name, "team_a": team_a_id, "team_b": team_b_id,
        "score_a": result["points_for"], "score_b": result["points_against"], "winner": winner
    }

    player_rows = []
    for tid, team_yards, team_score in [(team_a_id, result["total_yards"], result["points_for"]),
                                          (team_b_id, result["def_total_yards"], result["points_against"])]:
        pass_share = team_split["pass_share"].get(tid, team_split["pass_share"].mean())
        rush_share = 1 - pass_share
        team_pass_yards = team_yards * pass_share
        team_rush_yards = team_yards * rush_share
        team_rec_yards = team_pass_yards

        qb_rows = distribute_stats_capped_noisy_v2(team_pass_yards, current_qbs[current_qbs["team_id"] == tid], pass_shares, max_players=1, noise_std=0.2)
        rb_rows = distribute_stats_capped_noisy_v2(team_rush_yards, current_rbs[current_rbs["team_id"] == tid], rush_shares, max_players=2, noise_std=0.35)
        rec_rows = distribute_receiving_capped_noisy_v2(team_rec_yards,
                                                         current_wrs[current_wrs["team_id"] == tid],
                                                         current_tes[current_tes["team_id"] == tid],
                                                         wr_shares, te_shares, position_share, max_wr=4, max_te=2, noise_std=0.35)

        total_tds, total_fgs = estimate_tds_from_score_v5(team_score, tid)
        rush_td_share = team_td_counts["rush_td_share_correct"].get(tid, team_td_counts["rush_td_share_correct"].mean())
        rush_tds_target = round(total_tds * rush_td_share)
        pass_tds_target = total_tds - rush_tds_target

        ypc = rush_rates.loc[tid, "yards_per_carry"]
        ypt_catch = rec_rates.loc[tid, "yards_per_target"] / rec_rates.loc[tid, "catch_rate"]
        comp_pct = pass_rates.loc[tid, "comp_pct"]

        rb_rows = rb_rows.copy()
        if len(rb_rows) > 0:
            rb_rows["carries"] = rb_rows["simulated_stat"].apply(lambda y: max(round(y / ypc), 1) if ypc > 0 else 1)
            rb_weights = (rb_rows["simulated_stat"] / rb_rows["simulated_stat"].sum()).values
            rb_rows["tds"] = np.random.multinomial(rush_tds_target, rb_weights) if rush_tds_target > 0 else [0] * len(rb_rows)
        else:
            rb_rows["carries"], rb_rows["tds"] = [], []

        rec_rows = rec_rows.copy()
        if len(rec_rows) > 0:
            rec_rows["receptions"] = rec_rows["simulated_stat"].apply(lambda y: max(round(y / ypt_catch), 1) if ypt_catch > 0 else 1)
            rec_weights = (rec_rows["simulated_stat"] / rec_rows["simulated_stat"].sum()).values
            rec_rows["tds"] = np.random.multinomial(pass_tds_target, rec_weights) if pass_tds_target > 0 else [0] * len(rec_rows)
            total_rec_this = rec_rows["receptions"].sum()
        else:
            rec_rows["receptions"], rec_rows["tds"] = [], []
            total_rec_this = 0

        qb_rows = qb_rows.copy()
        if len(qb_rows) > 0:
            qb_rows["completions"] = total_rec_this
            qb_rows["attempts"] = max(round(total_rec_this / comp_pct), total_rec_this) if comp_pct > 0 else total_rec_this
            qb_rows["tds"] = pass_tds_target

        for df, stat_type in [(qb_rows, "passing"), (rb_rows, "rushing"), (rec_rows, "receiving")]:
            df = df.copy()
            df["stat_type"] = stat_type
            df["team_id"] = tid
            df["round"] = round_name
            player_rows.append(df)

    update_team_state(team_states, team_a_id, result)
    update_team_state(team_states, team_b_id, {
        "points_for": result["points_against"], "points_against": result["points_for"],
        "total_epa": result["def_total_epa"], "def_total_epa": result["total_epa"],
        "total_yards": result["def_total_yards"], "def_total_yards": result["total_yards"]
    })

    return winner, game_record, pd.concat(player_rows, ignore_index=True)

def simulate_conference_bracket_full(team_states, seeds_dict, conf_name, playoff_game_log, playoff_player_log):
    r1 = simulate_playoff_game_full(team_states, seeds_dict[2], seeds_dict[7], f"{conf_name} Wild Card")
    r2 = simulate_playoff_game_full(team_states, seeds_dict[3], seeds_dict[6], f"{conf_name} Wild Card")
    r3 = simulate_playoff_game_full(team_states, seeds_dict[4], seeds_dict[5], f"{conf_name} Wild Card")
    for winner, record, players in [r1, r2, r3]:
        playoff_game_log.append(record)
        playoff_player_log.append(players)

    wc_winners = [r1[0], r2[0], r3[0]]
    wc_seeds = {seeds_dict[2]: 2, seeds_dict[3]: 3, seeds_dict[4]: 4,
                seeds_dict[5]: 5, seeds_dict[6]: 6, seeds_dict[7]: 7}
    wc_winners_sorted = sorted(wc_winners, key=lambda t: wc_seeds[t])
    opponent_for_1 = wc_winners_sorted[-1]
    other_two = [t for t in wc_winners if t != opponent_for_1]

    d1 = simulate_playoff_game_full(team_states, seeds_dict[1], opponent_for_1, f"{conf_name} Divisional")
    if wc_seeds[other_two[0]] < wc_seeds[other_two[1]]:
        d2 = simulate_playoff_game_full(team_states, other_two[0], other_two[1], f"{conf_name} Divisional")
    else:
        d2 = simulate_playoff_game_full(team_states, other_two[1], other_two[0], f"{conf_name} Divisional")
    for winner, record, players in [d1, d2]:
        playoff_game_log.append(record)
        playoff_player_log.append(players)

    cc = simulate_playoff_game_full(team_states, d1[0], d2[0], f"{conf_name} Championship")
    playoff_game_log.append(cc[1])
    playoff_player_log.append(cc[2])
    return cc[0]

In [ ]:
# MONTE CARLO SIM

N_SIMULATIONS = 500

start_time = time.time()
all_season_results = []

for sim_num in range(N_SIMULATIONS):
    games_sim = simulate_one_season_lite(team_states, schedule_2026)
    games_sim["sim_id"] = sim_num
    all_season_results.append(games_sim)
    if (sim_num + 1) % 50 == 0:
        print(f"Completed {sim_num + 1}/{N_SIMULATIONS} simulations ({time.time() - start_time:.0f}s elapsed)")

all_seasons_df = pd.concat(all_season_results, ignore_index=True)
print(f"\nDone. {N_SIMULATIONS} simulations in {time.time() - start_time:.0f} seconds.")
print(all_seasons_df.shape)

Completed 50/500 simulations (86s elapsed)
Completed 100/500 simulations (170s elapsed)
Completed 150/500 simulations (253s elapsed)
Completed 200/500 simulations (335s elapsed)
Completed 250/500 simulations (418s elapsed)
Completed 300/500 simulations (500s elapsed)
Completed 350/500 simulations (582s elapsed)
Completed 400/500 simulations (665s elapsed)
Completed 450/500 simulations (748s elapsed)
Completed 500/500 simulations (830s elapsed)

Done. 500 simulations in 830 seconds.
(136000, 7)


In [21]:
home_results = all_seasons_df[["sim_id", "week", "home_id", "away_id", "home_score", "away_score"]].rename(
    columns={"home_id": "team_id", "away_id": "opp_id", "home_score": "team_score", "away_score": "opp_score"})
away_results = all_seasons_df[["sim_id", "week", "home_id", "away_id", "home_score", "away_score"]].rename(
    columns={"away_id": "team_id", "home_id": "opp_id", "away_score": "team_score", "home_score": "opp_score"})
all_team_games_sim = pd.concat([home_results, away_results], ignore_index=True)

all_team_games_sim["win"] = (all_team_games_sim["team_score"] > all_team_games_sim["opp_score"]).astype(int)
all_team_games_sim["loss"] = (all_team_games_sim["team_score"] < all_team_games_sim["opp_score"]).astype(int)
all_team_games_sim["tie"] = (all_team_games_sim["team_score"] == all_team_games_sim["opp_score"]).astype(int)

season_records = all_team_games_sim.groupby(["sim_id", "team_id"]).agg(
    wins=("win", "sum"), losses=("loss", "sum"), ties=("tie", "sum")
).reset_index()

season_records_full = season_records.merge(team_conf_div, on="team_id")
avg_points = all_team_games_sim.groupby(["sim_id", "team_id"])["team_score"].mean().reset_index().rename(columns={"team_score": "avg_points"})
season_records_full = season_records_full.merge(avg_points, on=["sim_id", "team_id"])

print(season_records_full.shape)

(16000, 8)


In [22]:
all_playoff_results = []
for sim_id in season_records_full["sim_id"].unique():
    sim_data = season_records_full[season_records_full["sim_id"] == sim_id]
    seeds = get_playoff_seeds(sim_data)
    seeds["sim_id"] = sim_id
    all_playoff_results.append(seeds)

all_playoffs_df = pd.concat(all_playoff_results, ignore_index=True)

playoff_rate = all_playoffs_df.groupby("team_id").size().reset_index(name="playoff_appearances")
playoff_rate["playoff_pct"] = round(playoff_rate["playoff_appearances"] / N_SIMULATIONS * 100, 1)
playoff_rate["team_abbr"] = playoff_rate["team_id"].map(team_abbr_lookup)
playoff_rate = playoff_rate.sort_values("playoff_pct", ascending=False)

print(playoff_rate.to_string(index=False))

 team_id  playoff_appearances  playoff_pct team_abbr
     325                  443         88.6       BAL
    4600                  353         70.6       SEA
    1400                  352         70.4       DEN
    3700                  338         67.6       PHI
    2250                  335         67.0       JAX
     610                  330         66.0       BUF
     920                  297         59.4       CIN
    1540                  286         57.2       DET
    2510                  282         56.4       LAR
    1800                  274         54.8        GB
    4900                  269         53.8        TB
    2120                  258         51.6       HOU
    3300                  217         43.4        NO
    2310                  200         40.0        KC
    5110                  200         40.0       WAS
     810                  190         38.0       CHI
    3200                  190         38.0        NE
    4500                  176         35.2    

In [23]:
start_time = time.time()
sb_results = []

for sim_id in season_records_full["sim_id"].unique():
    sim_seeds = all_playoffs_df[all_playoffs_df["sim_id"] == sim_id]
    if len(sim_seeds) != 14:  # safety check: skip malformed seeding
        continue
    sb_winner, afc_champ, nfc_champ = simulate_full_playoffs(team_states, sim_seeds)
    sb_results.append({"sim_id": sim_id, "sb_winner": sb_winner, "afc_champ": afc_champ, "nfc_champ": nfc_champ})

sb_results_df = pd.DataFrame(sb_results)
print(f"Simulated playoffs for {len(sb_results_df)} seasons in {time.time() - start_time:.1f} seconds")

Simulated playoffs for 500 seasons in 40.0 seconds


In [24]:
sb_winner_counts = sb_results_df["sb_winner"].value_counts().reset_index()
sb_winner_counts.columns = ["team_id", "sb_wins"]
sb_winner_counts["sb_win_pct"] = round(sb_winner_counts["sb_wins"] / len(sb_results_df) * 100, 1)

conf_champ_counts = pd.concat([sb_results_df["afc_champ"], sb_results_df["nfc_champ"]]).value_counts().reset_index()
conf_champ_counts.columns = ["team_id", "conf_champ_appearances"]
conf_champ_counts["conf_champ_pct"] = round(conf_champ_counts["conf_champ_appearances"] / len(sb_results_df) * 100, 1)

final_summary = playoff_rate[["team_id", "team_abbr", "playoff_pct"]].merge(
    conf_champ_counts[["team_id", "conf_champ_pct"]], on="team_id", how="left"
).merge(
    sb_winner_counts[["team_id", "sb_win_pct"]], on="team_id", how="left"
)
final_summary[["conf_champ_pct", "sb_win_pct"]] = final_summary[["conf_champ_pct", "sb_win_pct"]].fillna(0)
final_summary = final_summary.sort_values("sb_win_pct", ascending=False)

print(final_summary.to_string(index=False))

 team_id team_abbr  playoff_pct  conf_champ_pct  sb_win_pct
     325       BAL         88.6            20.0        11.4
     610       BUF         66.0            14.4         8.0
    1400       DEN         70.4            13.0         6.8
    4600       SEA         70.6            13.8         6.8
    2250       JAX         67.0            11.4         6.2
    2310        KC         40.0             6.8         5.2
    1540       DET         57.2            10.0         5.0
    3700       PHI         67.6            10.4         4.4
    2510       LAR         56.4            10.6         4.2
    4900        TB         53.8             7.4         4.2
    1800        GB         54.8             8.2         3.6
     920       CIN         59.4             7.2         3.2
    4500        SF         35.2             5.0         3.2
     810       CHI         38.0             5.2         2.6
    2120       HOU         51.6             6.0         2.4
    3300        NO         43.4         

In [25]:
final_summary.to_sql("season_simulation_summary", conn, if_exists="replace", index=False)
season_records.to_sql("season_records_by_sim", conn, if_exists="replace", index=False)
all_playoffs_df.to_sql("playoff_seeds_by_sim", conn, if_exists="replace", index=False)
sb_results_df.to_sql("playoff_results_by_sim", conn, if_exists="replace", index=False)
all_seasons_df.to_sql("all_simulated_games", conn, if_exists="replace", index=False)

print("Saved 5 Monte Carlo tables to nfl.db")
for t in ["season_simulation_summary", "season_records_by_sim", "playoff_seeds_by_sim",
          "playoff_results_by_sim", "all_simulated_games"]:
    count = pd.read_sql(f"SELECT COUNT(*) FROM {t}", conn).iloc[0, 0]
    print(f"  {t}: {count} rows")

Saved 5 Monte Carlo tables to nfl.db
  season_simulation_summary: 32 rows
  season_records_by_sim: 16000 rows
  playoff_seeds_by_sim: 7000 rows
  playoff_results_by_sim: 500 rows
  all_simulated_games: 136000 rows


In [26]:
# rebuild team_states completely fresh -- guarantees no contamination from earlier testing
team_states = init_team_states(team_games, all_team_ids)

games_final, players_final = simulate_one_season_full(team_states, schedule_2026)
print(games_final.shape, players_final.shape)

all_scores_check = pd.concat([games_final["home_score"], games_final["away_score"]])
print("League-wide average score (sanity check, should be ~22-23):", round(all_scores_check.mean(), 1))

(272, 10) (4896, 14)
League-wide average score (sanity check, should be ~22-23): 22.9


In [27]:
home_r = games_final[["home_id", "away_id", "home_score", "away_score"]].rename(
    columns={"home_id": "team_id", "away_id": "opp_id", "home_score": "team_score", "away_score": "opp_score"})
away_r = games_final[["home_id", "away_id", "home_score", "away_score"]].rename(
    columns={"away_id": "team_id", "home_id": "opp_id", "away_score": "team_score", "home_score": "opp_score"})
this_season_team_games = pd.concat([home_r, away_r], ignore_index=True)

this_season_team_games["win"] = (this_season_team_games["team_score"] > this_season_team_games["opp_score"]).astype(int)
this_season_team_games["loss"] = (this_season_team_games["team_score"] < this_season_team_games["opp_score"]).astype(int)
this_season_team_games["tie"] = (this_season_team_games["team_score"] == this_season_team_games["opp_score"]).astype(int)

this_season_record = this_season_team_games.groupby("team_id").agg(
    wins=("win", "sum"), losses=("loss", "sum"), ties=("tie", "sum")
).reset_index()
this_season_record = this_season_record.merge(team_conf_div, on="team_id")
avg_pts_this_season = this_season_team_games.groupby("team_id")["team_score"].mean().reset_index().rename(columns={"team_score": "avg_points"})
this_season_record = this_season_record.merge(avg_pts_this_season, on="team_id")

this_season_seeds = get_playoff_seeds(this_season_record)
this_season_seeds["conf"] = this_season_seeds["team_conf"]
print(this_season_seeds[["conf", "seed", "team_id"]].sort_values(["conf", "seed"]))

   conf  seed  team_id
11  AFC     1     2100
2   AFC     2      610
26  AFC     3     3900
27  AFC     4     4400
12  AFC     5     2120
20  AFC     6     3200
8   AFC     7     1400
16  NFC     1     2510
4   NFC     2      810
21  NFC     3     3300
31  NFC     4     5110
25  NFC     5     3800
28  NFC     6     4500
10  NFC     7     1800


In [28]:
# snapshot BEFORE playoffs, so the bracket can be re-run cleanly without re-simulating the regular season
team_states_pre_playoffs = copy.deepcopy(team_states)

playoff_game_log = []
playoff_player_log = []

afc_seeds_dict = this_season_seeds[this_season_seeds["conf"] == "AFC"].set_index("seed")["team_id"].to_dict()
nfc_seeds_dict = this_season_seeds[this_season_seeds["conf"] == "NFC"].set_index("seed")["team_id"].to_dict()

afc_champion = simulate_conference_bracket_full(team_states, afc_seeds_dict, "AFC", playoff_game_log, playoff_player_log)
nfc_champion = simulate_conference_bracket_full(team_states, nfc_seeds_dict, "NFC", playoff_game_log, playoff_player_log)

sb_winner, sb_record, sb_players = simulate_playoff_game_full(team_states, afc_champion, nfc_champion, "Super Bowl", neutral_site=True)
playoff_game_log.append(sb_record)
playoff_player_log.append(sb_players)

playoff_games_df = pd.DataFrame(playoff_game_log)
playoff_players_df = pd.concat(playoff_player_log, ignore_index=True)

print("AFC Champion:", team_abbr_lookup[afc_champion])
print("NFC Champion:", team_abbr_lookup[nfc_champion])
print("Super Bowl Winner:", team_abbr_lookup[sb_winner])

AFC Champion: SD
NFC Champion: LAR
Super Bowl Winner: LAR


In [29]:
games_final.to_sql("final_season_games", conn, if_exists="replace", index=False)
players_final.to_sql("final_season_players", conn, if_exists="replace", index=False)
this_season_seeds.to_sql("final_playoff_seeds", conn, if_exists="replace", index=False)
playoff_games_df.to_sql("final_playoff_games", conn, if_exists="replace", index=False)
playoff_players_df.to_sql("final_playoff_players", conn, if_exists="replace", index=False)

print("Saved 5 final tables.")
for t in ["final_season_games", "final_season_players", "final_playoff_seeds", "final_playoff_games", "final_playoff_players"]:
    count = pd.read_sql(f"SELECT COUNT(*) FROM {t}", conn).iloc[0, 0]
    print(f"  {t}: {count} rows")

Saved 5 final tables.
  final_season_games: 272 rows
  final_season_players: 4896 rows
  final_playoff_seeds: 14 rows
  final_playoff_games: 13 rows
  final_playoff_players: 234 rows


In [30]:
rams_id = abbr_to_id["LA"]
dolphins_id = abbr_to_id["MIA"]

print("=== MONTE CARLO SUMMARY (500 sims) ===")
print(final_summary[final_summary["team_id"].isin([rams_id, dolphins_id])].to_string(index=False))

print("\n=== Average simulated record across all 500 sims ===")
mc_avg_record = season_records_full[season_records_full["team_id"].isin([rams_id, dolphins_id])].groupby("team_id").agg(
    avg_wins=("wins", "mean"), avg_losses=("losses", "mean"), avg_ties=("ties", "mean")
).reset_index()
mc_avg_record["team_abbr"] = mc_avg_record["team_id"].map(team_abbr_lookup)
print(mc_avg_record.to_string(index=False))

=== MONTE CARLO SUMMARY (500 sims) ===
 team_id team_abbr  playoff_pct  conf_champ_pct  sb_win_pct
    2510       LAR         56.4            10.6         4.2
    2700       MIA         34.4             3.2         2.2

=== Average simulated record across all 500 sims ===
 team_id  avg_wins  avg_losses  avg_ties team_abbr
    2510     9.076       7.328     0.596       LAR
    2700     7.306       9.054     0.640       MIA


In [31]:
def print_team_season(team_id, games_df, team_abbr_lookup):
    abbr = team_abbr_lookup[team_id]
    team_games_this = games_df[(games_df["home_id"] == team_id) | (games_df["away_id"] == team_id)].sort_values("week")
    wins, losses, ties = 0, 0, 0
    for _, g in team_games_this.iterrows():
        is_home = g["home_id"] == team_id
        team_score = g["home_score"] if is_home else g["away_score"]
        opp_score = g["away_score"] if is_home else g["home_score"]
        opp_abbr = team_abbr_lookup[g["away_id"] if is_home else g["home_id"]]
        if team_score > opp_score:
            wins += 1; result_str = "W"
        elif team_score < opp_score:
            losses += 1; result_str = "L"
        else:
            ties += 1; result_str = "T"
        matchup = f"vs {opp_abbr}" if is_home else f"@ {opp_abbr}"
        print(f"Week {g['week']:>2}  {matchup:<8}  {abbr} {team_score:.0f} - {opp_score:.0f} {opp_abbr}  [{result_str}]")
    print(f"\nFinal record: {wins}-{losses}-{ties}\n")
    return team_games_this["game_id"].tolist()

print("=== RAMS 2026 SIMULATED SEASON ===")
rams_game_ids = print_team_season(rams_id, games_final, team_abbr_lookup)

print("=== DOLPHINS 2026 SIMULATED SEASON ===")
dolphins_game_ids = print_team_season(dolphins_id, games_final, team_abbr_lookup)

=== RAMS 2026 SIMULATED SEASON ===
Week  1  vs SF     LAR 33 - 31 SF  [W]
Week  2  vs NYG    LAR 33 - 13 NYG  [W]
Week  3  @ DEN     LAR 23 - 34 DEN  [L]
Week  4  @ PHI     LAR 21 - 19 PHI  [W]
Week  5  vs BUF    LAR 19 - 31 BUF  [L]
Week  6  vs ARI    LAR 29 - 24 ARI  [W]
Week  7  @ LV      LAR 28 - 28 LV  [T]
Week  8  vs SD     LAR 24 - 9 SD  [W]
Week  9  @ WAS     LAR 27 - 24 WAS  [W]
Week 10  @ ARI     LAR 28 - 23 ARI  [W]
Week 12  vs GB     LAR 27 - 16 GB  [W]
Week 13  vs KC     LAR 28 - 13 KC  [W]
Week 14  @ SF      LAR 42 - 23 SF  [W]
Week 15  vs DAL    LAR 24 - 15 DAL  [W]
Week 16  @ SEA     LAR 19 - 14 SEA  [W]
Week 17  @ TB      LAR 21 - 15 TB  [W]
Week 18  vs SEA    LAR 28 - 19 SEA  [W]

Final record: 14-2-1

=== DOLPHINS 2026 SIMULATED SEASON ===
Week  1  @ LV      MIA 23 - 10 LV  [W]
Week  2  @ SF      MIA 14 - 16 SF  [L]
Week  3  vs KC     MIA 20 - 21 KC  [L]
Week  4  @ MIN     MIA 16 - 35 MIN  [L]
Week  5  vs CIN    MIA 35 - 13 CIN  [W]
Week  7  @ NYJ     MIA 18 - 22 NYJ

In [32]:
import random

def print_box_score(game_id, games_df, players_df, team_abbr_lookup):
    g = games_df[games_df["game_id"] == game_id].iloc[0]
    home_abbr = team_abbr_lookup[g["home_id"]]
    away_abbr = team_abbr_lookup[g["away_id"]]
    print(f"=== {away_abbr} @ {home_abbr}  |  Final: {away_abbr} {g['away_score']:.0f} - {g['home_score']:.0f} {home_abbr} ===\n")
    for tid, abbr in [(g["home_id"], home_abbr), (g["away_id"], away_abbr)]:
        print(f"--- {abbr} ---")
        team_stats = players_df[(players_df["game_id"] == game_id) & (players_df["team_id"] == tid)]
        team_stats = team_stats[team_stats["simulated_stat"] >= 1]
        for stat_type in ["passing", "rushing", "receiving"]:
            rows = team_stats[team_stats["stat_type"] == stat_type].sort_values("simulated_stat", ascending=False)
            for _, r in rows.iterrows():
                if stat_type == "passing":
                    print(f"  {r['player_name']:<22} {r['completions']:.0f}/{r['attempts']:.0f}, {r['simulated_stat']:.0f} yds, {r['tds']:.0f} TD")
                elif stat_type == "rushing":
                    print(f"  {r['player_name']:<22} {r['carries']:.0f} car, {r['simulated_stat']:.0f} yds, {r['tds']:.0f} TD")
                elif stat_type == "receiving":
                    print(f"  {r['player_name']:<22} {r['receptions']:.0f} rec, {r['simulated_stat']:.0f} yds, {r['tds']:.0f} TD")
        print()

print("=== 3 RANDOM RAMS BOX SCORES ===\n")
for gid in random.sample(rams_game_ids, 3):
    print_box_score(gid, games_final, players_final, team_abbr_lookup)

print("=== 3 RANDOM DOLPHINS BOX SCORES ===\n")
for gid in random.sample(dolphins_game_ids, 3):
    print_box_score(gid, games_final, players_final, team_abbr_lookup)

=== 3 RANDOM RAMS BOX SCORES ===

=== NYG @ LAR  |  Final: NYG 13 - 33 LAR ===

--- LAR ---
  Matthew Stafford       24/37, 282 yds, 0 TD
  Kyren Williams         21 car, 85 yds, 0 TD
  Blake Corum            9 car, 36 yds, 0 TD
  Puka Nacua             8 rec, 95 yds, 0 TD
  Davante Adams          6 rec, 72 yds, 0 TD
  Colby Parkinson        4 rec, 43 yds, 0 TD
  Tyler Higbee           2 rec, 28 yds, 0 TD
  Jordan Whittington     2 rec, 25 yds, 0 TD
  Xavier Smith           2 rec, 18 yds, 0 TD

--- NYG ---
  Jaxson Dart            21/33, 216 yds, 1 TD
  Cam Skattebo           23 car, 102 yds, 0 TD
  Tyrone Tracy Jr.       5 car, 20 yds, 0 TD
  Malik Nabers           6 rec, 59 yds, 0 TD
  Darius Slayton         4 rec, 45 yds, 0 TD
  Isaiah Likely          3 rec, 33 yds, 0 TD
  Darnell Mooney         3 rec, 31 yds, 1 TD
  Calvin Austin III      3 rec, 26 yds, 0 TD
  Theo Johnson           2 rec, 21 yds, 0 TD

=== LAR @ TB  |  Final: LAR 21 - 15 TB ===

--- TB ---
  Baker Mayfield        